# TextCNN 验证集评估

加载验证损失最低的检查点，使用保存的词表评估验证集。测试集留到最终评估。

In [ ]:
# 加载检查点和验证集
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR, FIGURES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = CHECKPOINTS_DIR / "textcnn_extended_best.pt"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

if checkpoint["token_pattern"] != TOKEN_PATTERN.pattern or not checkpoint["lowercase"]:
    raise ValueError("Checkpoint tokenizer does not match the current sequence processing.")

validation_data = pd.read_csv(CLEANED_DIR / "val.csv")
validation_dataset = SentimentDataset(
    validation_data, checkpoint["vocabulary"], checkpoint["max_length"],
)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)

textcnn_model = TextCNN(**checkpoint["model_config"]).to(device)
textcnn_model.load_state_dict(checkpoint["model_state_dict"])
textcnn_model.eval()

print("Device:", device)
print(f"Checkpoint epoch: {checkpoint['epoch']}")
print(f"Validation samples: {len(validation_dataset):,}")

In [ ]:
# 对验证集进行预测
y_true = []
y_prediction = []

with torch.no_grad():
    for token_ids, lengths, labels in validation_loader:
        token_ids = token_ids.to(device)
        lengths = lengths.to(device)
        logits = textcnn_model(token_ids, lengths)

        y_true.extend(labels.tolist())
        y_prediction.extend(logits.argmax(dim=1).cpu().tolist())

print(f"Predictions: {len(y_prediction):,}")

In [ ]:
# 计算分类指标并与传统基线比较
textcnn_result = {
    "EXPERIMENT": "TextCNN",
    "FEATURES": "Token sequence",
    "MODEL": "TextCNN",
    **classification_metrics(y_true, y_prediction),
}

baseline_results = pd.read_csv(TABLES_DIR / "traditional_ml_validation_metrics.csv")
baseline_result = baseline_results[baseline_results["EXPERIMENT"] == "B06"]
comparison = pd.concat([baseline_result, pd.DataFrame([textcnn_result])], ignore_index=True)
display(comparison.round(4))

TABLES_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(TABLES_DIR / "textcnn_validation_comparison.csv", index=False)

In [ ]:
# 绘制验证集混淆矩阵
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_true, y_prediction, labels=[0, 1])

figure, axis = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues", ax=axis,
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
)
axis.set_xlabel("Predicted label")
axis.set_ylabel("True label")
axis.set_title("TextCNN Validation Confusion Matrix")
figure.tight_layout()

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "textcnn_validation_confusion_matrix.png", dpi=150)
plt.show()

## TextCNN 与 B06 的预测比较

在相同验证集上重新训练 B06，比较两个模型的预测是否正确，并查看预测分歧的评论。

In [ ]:
# 重新生成 B06 的验证集预测
import joblib
from sklearn.svm import LinearSVC
from sentiment_analysis.paths import TFIDF_DIR

train_data = pd.read_csv(CLEANED_DIR / "train.csv")
combined_vectorizer = joblib.load(TFIDF_DIR / "combined.joblib")
X_train_combined = combined_vectorizer.transform(train_data["text"])
X_validation_combined = combined_vectorizer.transform(validation_data["text"])

combined_svm_model = LinearSVC(C=1.0, max_iter=1000, random_state=42)
combined_svm_model.fit(X_train_combined, train_data["label"])
y_prediction_svm = combined_svm_model.predict(X_validation_combined)

print("B06 validation metrics:", classification_metrics(y_true, y_prediction_svm))

In [ ]:
# 统计两种模型的预测是否正确
prediction_comparison = validation_data[["text", "label"]].copy()
prediction_comparison["B06"] = y_prediction_svm
prediction_comparison["TextCNN"] = y_prediction

svm_correct = prediction_comparison["B06"] == prediction_comparison["label"]
textcnn_correct = prediction_comparison["TextCNN"] == prediction_comparison["label"]

comparison_summary = pd.DataFrame([
    {"CASE": "Both correct", "COUNT": (svm_correct & textcnn_correct).sum()},
    {"CASE": "Only B06 correct", "COUNT": (svm_correct & ~textcnn_correct).sum()},
    {"CASE": "Only TextCNN correct", "COUNT": (~svm_correct & textcnn_correct).sum()},
    {"CASE": "Both wrong", "COUNT": (~svm_correct & ~textcnn_correct).sum()},
])

display(comparison_summary)
comparison_summary.to_csv(TABLES_DIR / "textcnn_svm_prediction_comparison.csv", index=False)

In [ ]:
# 查看两个模型预测分歧的评论
print("Only B06 correct:")
display(prediction_comparison[svm_correct & ~textcnn_correct].head(3))

print("Only TextCNN correct:")
display(prediction_comparison[~svm_correct & textcnn_correct].head(3))

print("Both wrong:")
display(prediction_comparison[~svm_correct & ~textcnn_correct].head(3))

对部分样本的初步观察：

- Index 2 的标签为正面，评论同时包含赞赏和批评；B06 判断正确，TextCNN 误判。
- Index 45 的标签为负面，评论先肯定笑点和演员，后面着重表达不满；TextCNN 判断正确，B06 误判。
- Index 6 的标签为负面，评论同时批评电影并表达对导演的喜爱；两个模型都误判。
- Index 44 虽然标签为负面，但正文多数内容似乎在赞赏电影；TextCNN 与标签一致，仍值得进一步核对。

以上只是少量样本的观察，不能仅凭这些文字确定模型误判原因或推断总体规律。